In [0]:
# Cargar la tabla raw creada en el notebook anterior
from pyspark.sql.functions import col, sum as spark_sum, count, isnan, when

df = spark.table("wine_quality_raw")

print(f"Total de filas: {df.count()}")
print(f"Total de columnas: {len(df.columns)}")

Total de filas: 6497
Total de columnas: 13


In [0]:
# Revisar valores nulos en cada columna
null_counts = df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])
display(null_counts)

fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol,quality,wine_type
0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
# Detectar y eliminar duplicados
total = df.count()
distinct = df.distinct().count()
duplicates = total - distinct

print(f"Filas totales:    {total}")
print(f"Filas únicas:     {distinct}")
print(f"Duplicados:       {duplicates}")

df_clean = df.dropDuplicates()

Filas totales:    6497
Filas únicas:     5320
Duplicados:       1177


In [0]:
# Estadísticas descriptivas básicas
display(df_clean.describe())

summary,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol,quality,wine_type
count,5320,5320,5320,5320,5320,5320,5320,5320,5320,5320,5320,5320,5320
mean,7.21517857142857,0.34412969924812054,0.31849436090225286,5.048477443608999,0.056689849624059784,30.036654135338345,114.10902255639098,0.9945352988721753,3.224663533834598,0.5333571428571398,10.549241228070208,5.795676691729323,null
stddev,1.3196706812561014,0.16824826021320788,0.14715733429246616,4.500180119377678,0.036863314781293406,17.805044757279468,56.77422320849698,0.002965505063396257,0.16037920243624498,0.14974292781635923,1.1859329282542304,0.8797721170771579,null
min,3.8,0.08,0.0,0.6,0.009,1.0,6.0,0.98711,2.72,0.22,8.0,3,red
max,15.9,1.58,1.66,65.8,0.611,289.0,440.0,1.03898,4.01,2.0,14.9,9,white


In [0]:
# Revisar el rango de 'quality' (debe ser 0-10)
display(df_clean.groupBy("quality").count().orderBy("quality"))

quality,count
3,30
4,206
5,1752
6,2323
7,856
8,148
9,5


In [0]:
# Crear columna de categoría de calidad
# Esta columna se usará en el análisis y el dashboard
df_clean = df_clean.withColumn(
    "quality_category",
    when(col("quality") >= 7, "Alta")
    .when(col("quality") >= 5, "Media")
    .otherwise("Baja")
)

In [0]:
# Guardar tabla limpia
df_clean.write.format("delta").mode("overwrite").saveAsTable("wine_quality_clean")

print(f"✅ Tabla 'wine_quality_clean' guardada con {df_clean.count()} filas.")
display(df_clean.limit(5))

✅ Tabla 'wine_quality_clean' guardada con 5320 filas.


fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol,quality,wine_type,quality_category
6.2,0.32,0.45,2.9,0.029,37.0,94.0,0.98998,3.25,0.6,12.4,6,white,Media
6.9,0.19,0.38,1.15,0.023,30.0,105.0,0.99047,3.11,0.38,11.4,5,white,Media
7.0,0.2,0.4,1.1,0.058,30.0,93.0,0.99322,3.03,0.38,9.2,6,white,Media
6.5,0.21,0.42,1.1,0.059,33.0,101.0,0.9927,3.12,0.38,9.7,6,white,Media
6.8,0.27,0.24,4.6,0.098,36.0,127.0,0.99412,3.15,0.49,9.6,6,white,Media
